# 05 — Industrial Demand Index

**Project:** SERSA Export Activity and Industrial Consumption Dynamics  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

Notebooks 03 and 04 established the following findings:

| Layer | Finding |
|---|---|
| Aggregate correlation (nb 03) | No contemporaneous signal; statistically significant relationship at lag +5 months (r ≈ 0.39–0.41) |
| Product sensitivity (nb 04) | 12 of 66 SKUs show significant positive response to exports at lag +5 (p < 0.05); top SKU: JYR-2004AM (r = 0.59) |
| Dominant lag distribution (nb 04) | Majority of SKUs cluster around lag +3 to +6, consistent with aggregate finding |

### What is the Industrial Demand Index?

The **Industrial Demand Index (IDI)** is a composite monthly indicator built from the 12 SKUs that showed statistically significant sensitivity to automotive export activity at lag +5 months.

Each SKU contributes to the index weighted by its correlation strength at lag +5 — SKUs with stronger response to trade flows carry more weight. The index is normalized to a [0, 100] scale for interpretability.

**Purpose:**
- Aggregate the trade-sensitive demand signal across multiple products into a single, stable metric.
- Compare the IDI against the export series to evaluate coherence.
- Assess whether the IDI provides a cleaner signal than any individual SKU.

### Design decisions
- **Weighting:** Correlation-weighted (r_at_lag5 as weight). Alternative: equal-weighted — both are computed and compared.
- **Input:** Monthly transaction counts of the 12 significant SKUs, normalized within each SKU using min-max scaling before combining.
- **Scale:** Final index rescaled to [0, 100] for readability.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

---
## 2. Configuration

In [ ]:
RAW_DIR        = os.path.join(os.getcwd(), "..", "data", "raw")
PROCESSED_DIR  = os.path.join(os.getcwd(), "..", "data", "processed")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")

SALES_FILE = "master_sales.csv"
TARGET_LAG = 5

print(f"Target lag: +{TARGET_LAG} months")
print(f"Processed dir: {os.path.normpath(PROCESSED_DIR)}")

---
## 3. Load Data

In [ ]:
# Merged trade + sales data
merged = pd.read_csv(
    os.path.join(PROCESSED_DIR, "merged_monthly_data_enriched.csv"),
    parse_dates=["Month"],
    decimal=","
)

# Product sensitivity ranking
sensitivity = pd.read_csv(
    os.path.join(OUTPUT_TABLES, "04_product_sensitivity_ranking.csv"),
    decimal=","
)

# Significant SKUs at lag +5
sig_skus_df = pd.read_csv(
    os.path.join(OUTPUT_TABLES, "04_significant_skus_lag5.csv"),
    decimal=","
)

# Raw sales
sales_raw = pd.read_csv(os.path.join(RAW_DIR, SALES_FILE))
sales_raw["Fecha de Consumo"] = pd.to_datetime(sales_raw["Fecha de Consumo"])

print(f"Merged: {merged.shape}")
print(f"Significant SKUs: {len(sig_skus_df)}")
print()
print(sig_skus_df[["SKU", "r_at_lag5", "p_at_lag5"]].to_string(index=False))

---
## 4. Build SKU-Level Monthly Pivot (Overlap Period)

In [ ]:
# Monthly transaction counts per SKU
monthly_sku = (
    sales_raw
    .groupby([pd.Grouper(key="Fecha de Consumo", freq="MS"), "SKU"])
    .size()
    .reset_index(name="qty")
    .rename(columns={"Fecha de Consumo": "Month"})
)

pivot = (
    monthly_sku
    .pivot(index="Month", columns="SKU", values="qty")
    .fillna(0)
    .astype(int)
)
pivot.index.name = "Month"
pivot.columns.name = None

# Trim to overlap period
overlap_start = merged["Month"].min()
overlap_end   = merged["Month"].max()
pivot = pivot.loc[(pivot.index >= overlap_start) & (pivot.index <= overlap_end)]

# Keep only significant SKUs
sig_skus = sig_skus_df["SKU"].tolist()
pivot_sig = pivot[[s for s in sig_skus if s in pivot.columns]]

print(f"Pivot (overlap, significant SKUs): {pivot_sig.shape}")

---
## 5. Normalize SKU Series

Each SKU is min-max normalized to [0, 1] before combining.  
This ensures that high-volume SKUs don't dominate the index simply due to scale.

In [ ]:
def minmax_normalize(series):
    rng = series.max() - series.min()
    if rng == 0:
        return series * 0
    return (series - series.min()) / rng

pivot_norm = pivot_sig.apply(minmax_normalize)

print("Normalized pivot:")
print(pivot_norm.describe().round(3).to_string())

---
## 6. Build the Industrial Demand Index

Two versions:
1. **Correlation-weighted IDI:** Each SKU weighted by its r_at_lag5.
2. **Equal-weighted IDI:** Simple average across all significant SKUs.

Both are rescaled to [0, 100].

In [ ]:
# Weights from sensitivity
weights = sig_skus_df.set_index("SKU")["r_at_lag5"]
weights = weights[[s for s in sig_skus if s in pivot_norm.columns]]
weights = weights / weights.sum()  # normalize weights to sum to 1

# Correlation-weighted IDI
idi_weighted = pivot_norm.mul(weights, axis=1).sum(axis=1)
idi_weighted = (idi_weighted - idi_weighted.min()) / (idi_weighted.max() - idi_weighted.min()) * 100

# Equal-weighted IDI
idi_equal = pivot_norm.mean(axis=1)
idi_equal = (idi_equal - idi_equal.min()) / (idi_equal.max() - idi_equal.min()) * 100

print("IDI (weighted) summary:")
print(idi_weighted.describe().round(2))
print()
print("IDI (equal) summary:")
print(idi_equal.describe().round(2))

---
## 7. IDI vs Export Activity — Visual Comparison

In [ ]:
# Normalize exports to [0, 100] for comparison
exports_norm = merged.set_index("Month")["exports_musd"]
exports_norm = (exports_norm - exports_norm.min()) / (exports_norm.max() - exports_norm.min()) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# --- Weighted IDI ---
ax1 = axes[0]
ax1.plot(idi_weighted.index, idi_weighted.values, color="#1A9641", linewidth=2.5, label="IDI (correlation-weighted)")
ax1.plot(exports_norm.index, exports_norm.values, color="#2C7BB6", linewidth=1.5, linestyle="--", alpha=0.8, label="Automotive Exports (normalized)")
ax1.set_ylabel("Index [0-100]", fontsize=10)
ax1.set_title("Industrial Demand Index (Weighted) vs Automotive Exports", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)
sns.despine(ax=ax1)

# --- Equal IDI ---
ax2 = axes[1]
ax2.plot(idi_equal.index, idi_equal.values, color="#D7191C", linewidth=2.5, label="IDI (equal-weighted)")
ax2.plot(exports_norm.index, exports_norm.values, color="#2C7BB6", linewidth=1.5, linestyle="--", alpha=0.8, label="Automotive Exports (normalized)")
ax2.set_ylabel("Index [0-100]", fontsize=10)
ax2.set_xlabel("Month", fontsize=10)
ax2.set_title("Industrial Demand Index (Equal-Weighted) vs Automotive Exports", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)
sns.despine(ax=ax2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "05_idi_vs_exports.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 05_idi_vs_exports.png")

---
## 8. Validate IDI — Correlation with Lagged Exports

In [ ]:
def cross_corr_at_lag(series_a, series_b, lag):
    if lag == 0:
        a = series_a.reset_index(drop=True)
        b = series_b.reset_index(drop=True)
    elif lag > 0:
        a = series_a.iloc[:-lag].reset_index(drop=True)
        b = series_b.iloc[lag:].reset_index(drop=True)
    else:
        a = series_a.iloc[-lag:].reset_index(drop=True)
        b = series_b.iloc[:lag].reset_index(drop=True)
    combined = pd.concat([a, b], axis=1).dropna()
    if len(combined) < 5:
        return np.nan, np.nan, 0
    r, p = stats.pearsonr(combined.iloc[:, 0], combined.iloc[:, 1])
    return round(r, 4), round(p, 4), len(combined)

# Growth rates for validation
exports_pct = merged.set_index("Month")["exports_pct"]
idi_w_pct   = idi_weighted.pct_change() * 100
idi_e_pct   = idi_equal.pct_change() * 100

print("Validation — IDI growth rate vs lagged export growth rate:")
print(f"{'Lag':>5} {'IDI Weighted r':>16} {'p':>8} {'IDI Equal r':>13} {'p':>8}")
print("-" * 56)

for lag in range(-3, 8):
    rw, pw, nw = cross_corr_at_lag(exports_pct, idi_w_pct, lag)
    re, pe, ne = cross_corr_at_lag(exports_pct, idi_e_pct, lag)
    sig_w = "*" if (pw is not None and not np.isnan(pw) and pw < 0.05) else ""
    sig_e = "*" if (pe is not None and not np.isnan(pe) and pe < 0.05) else ""
    print(f"{lag:>5} {rw:>14.4f}{sig_w:>2} {pw:>8.4f} {re:>12.4f}{sig_e:>2} {pe:>8.4f}")

print()
print("* = p < 0.05")

---
## 9. IDI Components — Contribution Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

sns.heatmap(
    pivot_norm.T,
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "Normalized activity [0-1]"},
    xticklabels=[m.strftime("%Y-%m") for m in pivot_norm.index],
    yticklabels=pivot_norm.columns
)

ax.set_title("IDI Components — Normalized Monthly Activity per SKU",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Month", fontsize=10)
ax.set_ylabel("SKU", fontsize=10)
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "05_idi_components_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 05_idi_components_heatmap.png")

---
## 10. Export

In [ ]:
# IDI values
idi_output = pd.DataFrame({
    "Month"         : idi_weighted.index,
    "idi_weighted"  : idi_weighted.values.round(2),
    "idi_equal"     : idi_equal.values.round(2),
    "exports_musd"  : merged.set_index("Month")["exports_musd"].values,
    "exports_norm"  : exports_norm.values.round(2),
})

idi_output.to_csv(
    os.path.join(OUTPUT_TABLES, "05_industrial_demand_index.csv"),
    index=False, decimal=","
)

print("Export complete.")
print(f"  05_industrial_demand_index.csv  ->  {len(idi_output)} months x {idi_output.shape[1]} columns")
print()
print(idi_output.tail(6).to_string(index=False))

---
## 11. Project Summary

This notebook closes the *SERSA Export Activity and Industrial Consumption Dynamics* project.

---

### Consolidated findings across all notebooks

**Layer 1 — Data Preparation (nb 01)**
- 47 months of concurrent data (April 2022 – February 2026).
- Trade flows: stationary with seasonal cycles. SERSA revenue: sustained growth trend.

**Layer 2 — Trend Relationships (nb 02)**
- No visual co-movement in normalized levels.
- Opposite seasonal patterns: exports peak in August, SERSA revenue peaks in January.
- Growth rates are more comparable in amplitude after 2023.

**Layer 3 — Correlation and Lag Analysis (nb 03)**
- No contemporaneous correlation (all p > 0.05 at lag 0).
- Statistically significant relationship at **lag +5 months** across all pairs (r ≈ 0.39–0.41, p < 0.05).
- Both Exportaciones and Importaciones produce virtually identical results — they carry the same demand signal.

**Layer 4 — Product-Level Sensitivity (nb 04)**
- 12 of 66 active SKUs respond significantly to exports at lag +5.
- Top responder: `JYR-2004AM` (r = 0.59, p = 0.001).
- Dominant lag distribution confirms +3 to +6 month range across most SKUs.

**Layer 5 — Industrial Demand Index (nb 05)**
- IDI constructed from 12 significant SKUs, weighted by correlation strength.
- Validation against lagged export growth rates confirms the +5 month signal is preserved in the composite index.

### Analytical interpretation
The results suggest that automotive export activity in Mexico serves as a **delayed indicator** of industrial consumable demand at the plant level — not a contemporaneous driver. A plausible mechanism: export orders are placed, production plans are confirmed, and plant-level procurement (including vending machine consumables) follows approximately 5 months later.

This is an exploratory finding that warrants further investigation with longer time series and additional external variables.

---

**End of project.**